In [1]:
import numpy as np
import joblib
from utils import train_categorical_model, prepare_datasets, calibrate_incident_agent, train_incident_agent, generate_oof_features, find_optimal_threshold,predict_with_normal_penalty
from rcf_model import RCF
from meta_scorer import train_fusion_meta_learner, CostSensitiveMetaLearner, _incident_entropy, _top2_margin
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from soar_zta import ZeroTrustSOARAgent
from playbook_editor import PlaybookEditorAgent
import os
import datetime
from sklearn.preprocessing import RobustScaler

c:\Users\Arib Ansari\Desktop\Pithun\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = r"UNSW-NB15\unsw_train.csv"

# FIX (PCA Leakage): prepare_datasets now returns X_num_raw (6th value) —
# the raw, unscaled numerical frame. This is passed to generate_oof_features
# so each fold fits its own scaler+PCA exclusively on its training split.
# X_train_num (PCA-transformed using the full-train artifact) is kept for
# the final RCF training step in cell 5.
X_train_cat, X_train_num, X_train_num_raw, y_bin, cat_cols, y_multi = prepare_datasets(file_path, is_train=True)

Loading and transforming data from UNSW-NB15\unsw_train.csv (Mode: Train)...
Final Categorical features: 6
Numerical features processed: 20 Principal Components


In [ ]:
# FIX (PCA Leakage): Pass X_train_num_raw instead of X_train_num.
# generate_oof_features will fit a fresh scaler+PCA per fold internally.
# FIX: Train the final RCF model FIRST to establish the global mathematical anchors
print("\n--- Establishing Global RCF Anchors (Attack-Trained) ---")
final_rcf = RCF(num_trees=80, tree_size=256)
y_bin_reset = y_bin.reset_index(drop=True)

# RobustScaler fitted on attack-only rows.
# Median/IQR scaling is resistant to UNSW-NB15's extreme outliers.
rcf_scaler = RobustScaler()
X_num_train_scaled = rcf_scaler.fit_transform(
    X_train_num_raw[y_bin_reset == 1]   # ← attack rows only
)
final_rcf.rcf_scaler = rcf_scaler
final_rcf.fit_predict(X_num_train_scaled, global_p1=None, global_p99=None)
global_p1  = final_rcf._score_p1
global_p99 = final_rcf._score_p99
global_p995 = final_rcf._score_p995
final_rcf.save_model(r"Saves/rcf_base.pkl")

Loading and transforming data from UNSW-NB15\unsw_train.csv (Mode: Train)...
Final Categorical features: 6
Numerical features processed: 20 Principal Components

--- Establishing Global RCF Anchors (Attack-Trained) ---
Building RRCF Forest (80 trees)...


RRCF Training & Scoring: 100%|██████████| 45332/45332 [15:53<00:00, 47.56it/s] 


[RCF] Calibration anchors — p1=1.6328  p99=131.9952  p99.5=174.6569 (effective ceiling)  n_samples=45332
RCF state successfully saved to Saves/rcf_base.pkl


In [ ]:
# ── RCF STANDALONE DIAGNOSTIC ─────────────────────────────────────────────
# Run this cell immediately after the RCF is trained on the full training set.
# Shows exactly how much of the pipeline's performance comes from RCF alone,
# so you can isolate whether RCF or CatBoost is the weaker contributor.
#
# Requires: rcf_model, X_num_test, y_test_bin, optimal_threshold
# (all should already exist in your notebook namespace at this point)
# ──────────────────────────────────────────────────────────────────────────

test_file_path = r"UNSW-NB15\unsw_test.csv"

# Test mode: prepare_datasets returns X_num already transformed by the saved
# full-train scaler+PCA. X_num_raw is unused in the test path but unpacked
# for API consistency.
X_test_cat, X_test_num, X_test_num_raw, y_test_bin, cat_cols_test, y_test_multi = prepare_datasets(test_file_path, is_train=False)

# ── 1. Get raw RCF anomaly scores on the test set ─────────────────────────
X_test_num_scaled = final_rcf.rcf_scaler.transform(X_test_num_raw)
rcf_scores = 1.0 - final_rcf.predict_proba(X_test_num_scaled)

best_t, best_cost = 0.5, np.inf
for t in np.linspace(0.01, 0.99, 1000):
    preds = (rcf_scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_bin, preds).ravel()
    cost = (fn * 12) + (fp * 2)
    if cost < best_cost:
        best_cost, best_t = cost, t
 
best_preds = (rcf_scores >= best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test_bin, best_preds).ravel()

print(f"ROC-AUC : {roc_auc_score(y_test_bin, rcf_scores):.4f}")
print(f"\nRCF standalone (best threshold={best_t:.4f}):")
print(classification_report(y_test_bin, best_preds, target_names=["Normal", "Attack"]))
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"SOC cost: {best_cost}")

normal_scores = rcf_scores[y_test_bin == 0]
attack_scores = rcf_scores[y_test_bin == 1]
print(f"\nNormal scores — mean={normal_scores.mean():.4f}  p95={np.percentile(normal_scores,95):.4f}")
print(f"Attack scores — mean={attack_scores.mean():.4f}  p05={np.percentile(attack_scores,5):.4f}")
 
normal_scores = rcf_scores[y_test_bin == 0]
attack_scores = rcf_scores[y_test_bin == 1]
print(f"\nNormal scores — mean={normal_scores.mean():.4f}  p95={np.percentile(normal_scores,95):.4f}")
print(f"Attack scores — mean={attack_scores.mean():.4f}  p05={np.percentile(attack_scores,5):.4f}")


Loading and transforming data from UNSW-NB15\unsw_test.csv (Mode: Test)...
Final Categorical features: 6
Numerical features processed: 20 Principal Components


RRCF Blind Test Scoring: 100%|██████████| 175341/175341 [1:19:13<00:00, 36.89it/s]



[RCF] Score distribution — min=0.0054  mean=0.3169  max=1.0000  % at ceiling (≥0.999): 10.2%
[RCF] WARNING: 10.2% of scores saturated at ceiling. Expected ~0.5% (p99.5 ceiling). Consider retraining with more attack samples or checking for distribution shift between train and test.
ROC-AUC : 0.7471

RCF standalone (best threshold=0.0100):
              precision    recall  f1-score   support

      Normal       0.78      0.25      0.38     56000
      Attack       0.73      0.97      0.83    119341

    accuracy                           0.74    175341
   macro avg       0.76      0.61      0.61    175341
weighted avg       0.75      0.74      0.69    175341

TN=14233  FP=41767  FN=3960  TP=115381
SOC cost: 131054

Normal scores — mean=0.4873  p95=0.9622
Attack scores — mean=0.7750  p05=0.2944

Normal scores — mean=0.4873  p95=0.9622
Attack scores — mean=0.7750  p05=0.2944


In [4]:
oof_cat_scores, oof_rcf_scores, oof_incident_entropy, oof_top2_margin = generate_oof_features(
    X_train_cat, X_train_num_raw, y_bin, cat_cols,
    train_cat_func=train_categorical_model,
    rcf_class=RCF,
    train_incident_func=train_incident_agent,
    n_splits=5,
    y_multi=y_multi,
    rcf_num_trees=80, 
    rcf_tree_size=256,
    global_p1=global_p1,
    global_p99=global_p99,
    global_p995=global_p995
)


Generating 5-Fold OOF Predictions for Meta-Learner training...
  -> Processing Fold 1/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (80 trees)...


RRCF Training & Scoring: 100%|██████████| 36265/36265 [13:33<00:00, 44.59it/s] 


[RCF] Calibration anchors — p1=1.6328  p99=131.9952  p99.5=174.6569 (effective ceiling)  n_samples=36265


RRCF Blind Test Scoring: 100%|██████████| 16467/16467 [08:43<00:00, 31.46it/s]



[RCF] Score distribution — min=0.0001  mean=0.2868  max=1.0000  % at ceiling (≥0.999): 8.3%
[RCF] WARNING: 8.3% of scores saturated at ceiling. Expected ~0.5% (p99.5 ceiling). Consider retraining with more attack samples or checking for distribution shift between train and test.

Training Incident Agent (Multi-class Threat Classifier)...

Calibrating Incident Agent probabilities (isotonic regression)...
Calibration complete.
  -> Processing Fold 2/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (80 trees)...


...


[RCF] Score distribution — min=0.0000  mean=0.2810  max=1.0000  % at ceiling (≥0.999): 8.0%
[RCF] WARNING: 8.0% of scores saturated at ceiling. Expected ~0.5% (p99.5 ceiling). Consider retraining with more attack samples or checking for distribution shift between train and test.

Training Incident Agent (Multi-class Threat Classifier)...

Calibrating Incident Agent probabilities (isotonic regression)...
Calibration complete.
  -> Processing Fold 5/5...

Training CatBoost (with High Regularization)...
Building RRCF Forest (80 trees)...


RRCF Training & Scoring: 100%|██████████| 36266/36266 [13:09<00:00, 45.94it/s] 


[RCF] Calibration anchors — p1=1.6328  p99=131.9952  p99.5=174.6569 (effective ceiling)  n_samples=36266


RRCF Blind Test Scoring: 100%|██████████| 16466/16466 [07:40<00:00, 35.77it/s]



[RCF] Score distribution — min=0.0000  mean=0.2903  max=1.0000  % at ceiling (≥0.999): 8.3%
[RCF] WARNING: 8.3% of scores saturated at ceiling. Expected ~0.5% (p99.5 ceiling). Consider retraining with more attack samples or checking for distribution shift between train and test.

Training Incident Agent (Multi-class Threat Classifier)...

Calibrating Incident Agent probabilities (isotonic regression)...
Calibration complete.
OOF Generation Complete.


In [ ]:
X_meta_train_clean = np.column_stack((oof_cat_scores, oof_rcf_scores, oof_incident_entropy, oof_top2_margin))

# FIX (Issue 3 — min_precision raised to 0.80):
# COST_FN raised from 10 → 12 to maintain recall pressure while the higher
# precision floor pushes the threshold upward.  At 80% precision, roughly 1 in
# 5 flagged events may be a false alarm — acceptable for a SOC with automated
# response.  The weight-share check inside train_fusion_meta_learner will warn
# if CatBoost's contribution drops below 30% of total weight, which would
# indicate over-reliance on RCF/entropy and degraded detection of novel attacks.
COST_FN = 12
COST_FP = 2

meta_learner = train_fusion_meta_learner(
    X_meta_train_clean,
    y_bin,
    COST_FN=COST_FN,
    COST_FP=COST_FP,
    lambda_reg=0.1
)

meta_learner.save_model(r"Saves/meta_learner.pkl")

oof_meta_scores = meta_learner.predict_proba(X_meta_train_clean)
optimal_threshold = find_optimal_threshold(
    y_bin,
    oof_meta_scores,
    cost_fn=COST_FN,
    cost_fp=COST_FP,
    min_precision=0.75,   # lowered from 0.80 — pairs with min_recall to prevent threshold climbing too high
    min_recall=0.83,      # ensures at least 80% of attacks are caught (guards against high-FN collapse)
    save_path="Saves/optimal_threshold.json"
)



Training Meta-Learner (FN Penalty: 12, FP Penalty: 2, L2: 0.1)...

Meta-Learner weights — CatBoost: 4.6016  RCF: 0.2753  Entropy: 1.0707  Top2Margin: 1.3635  Bias: 2.5130
CatBoost weight share: 62.9% — healthy contribution.
Meta-Learner Training Complete!
💾 Model state successfully saved to Saves/meta_learner.pkl

--- THRESHOLD OPTIMISATION (OOF) ---
Sweep range    : 0.01 - 0.99 (1000 candidates, 935 skipped due to constraints)
Cost function  : (FN x 12) + (FP x 2)
Constraints    : Precision ≥ 0.75, Recall ≥ 0.83
Optimal threshold    : 0.7889
Precision @ threshold: 0.7504
Recall @ threshold   : 0.9856
Minimum SOC cost     : 37532
  TN=22140  FP=14860  FN=651  TP=44681
Threshold saved to Saves/optimal_threshold.json


In [3]:
# Train full-dataset base models (no OOF needed here — these are frozen for inference)
catboost_model = train_categorical_model(X_train_cat, y_bin, cat_cols)
print("\nPreparing Incident Agent calibration split...")
X_cat_final_tr, X_cat_final_cal, y_multi_final_tr, y_multi_final_cal = train_test_split(
    X_train_cat, y_multi, test_size=0.10, random_state=42, stratify=y_multi
)
# Train the base model on the 90% split
base_incident_model = train_incident_agent(X_cat_final_tr, y_multi_final_tr, cat_cols)

# Fit the calibrator on the 10% holdout split
calibrated_incident_model = calibrate_incident_agent(
    base_incident_model, 
    X_cat_final_cal, 
    y_multi_final_cal
)

print("\nSaving Base Models to disk...")
catboost_model.save_model(r"Saves/catboost_base.cbm")
joblib.dump(calibrated_incident_model, r"Saves/incident_calibrated.pkl") #just to avoid any inconsistencies with naming


Training CatBoost (with High Regularization)...

Preparing Incident Agent calibration split...

Training Incident Agent (Multi-class Threat Classifier)...

Calibrating Incident Agent probabilities (isotonic regression)...
Calibration complete.

Saving Base Models to disk...


['Saves/incident_calibrated.pkl']

In [4]:
loaded_catboost = CatBoostClassifier()
loaded_catboost.load_model("Saves/catboost_base.cbm")
loaded_incident = joblib.load("Saves/incident_calibrated.pkl")

cache = joblib.load("Saves/test_evaluation_cache.joblib")
X_test_cat          = cache["X_test_cat"]
y_test_bin          = cache["y_test_bin"]

# ── Load models ───────────────────────────────────────────────────────────
loaded_catboost = CatBoostClassifier()
loaded_catboost.load_model("Saves/catboost_base.cbm")
loaded_incident = joblib.load("Saves/incident_calibrated.pkl")

COST_FN = 12
COST_FP = 2

# ══════════════════════════════════════════════════════════════════════════
# 1. CATBOOST BINARY CLASSIFIER
# ══════════════════════════════════════════════════════════════════════════
cat_scores = loaded_catboost.predict_proba(X_test_cat)[:, 1]

best_t, best_cost = 0.5, np.inf
for t in np.linspace(0.01, 0.99, 1000):
    preds = (cat_scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_bin, preds).ravel()
    cost = (fn * COST_FN) + (fp * COST_FP)
    if cost < best_cost:
        best_cost, best_t = cost, t

best_preds = (cat_scores >= best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test_bin, best_preds).ravel()

print("=" * 60)
print("CATBOOST BINARY CLASSIFIER — STANDALONE")
print("=" * 60)
print(f"ROC-AUC : {roc_auc_score(y_test_bin, cat_scores):.4f}")
print(f"Best threshold : {best_t:.4f}  |  SOC cost : {best_cost}")
print(classification_report(y_test_bin, best_preds, target_names=["Normal", "Attack"]))
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

normal_s = cat_scores[y_test_bin == 0]
attack_s = cat_scores[y_test_bin == 1]
print(f"\nNormal scores — mean={normal_s.mean():.4f}  p95={np.percentile(normal_s, 95):.4f}")
print(f"Attack scores — mean={attack_s.mean():.4f}  p05={np.percentile(attack_s, 5):.4f}")

# ══════════════════════════════════════════════════════════════════════════
# 2. INCIDENT AGENT — MULTICLASS THREAT CLASSIFIER
# ══════════════════════════════════════════════════════════════════════════
incident_proba  = loaded_incident.predict_proba(X_test_cat)
incident_preds = predict_with_normal_penalty(loaded_incident, X_test_cat, normal_penalty=0.15)  # multiclass labels
class_names     = loaded_incident.classes_

# Collapse to binary: anything not "Normal" is an attack
incident_binary = (incident_preds != "Normal").astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_bin, incident_binary).ravel()
soc_cost = (fn * COST_FN) + (fp * COST_FP)

print("\n" + "=" * 60)
print("INCIDENT AGENT — MULTICLASS (collapsed to binary)")
print("=" * 60)
print(f"SOC cost (binary collapse) : {soc_cost}")
print(classification_report(y_test_bin, incident_binary, target_names=["Normal", "Attack"]))
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

print("\nPer-class prediction counts:")
unique, counts = np.unique(incident_preds, return_counts=True)
for cls, cnt in sorted(zip(unique, counts), key=lambda x: -x[1]):
    print(f"  {cls:<20} : {cnt:>7}  ({cnt/len(incident_preds)*100:.1f}%)")

CATBOOST BINARY CLASSIFIER — STANDALONE
ROC-AUC : 0.9684
Best threshold : 0.0404  |  SOC cost : 26668
              precision    recall  f1-score   support

      Normal       0.99      0.79      0.88     56000
      Attack       0.91      1.00      0.95    119341

    accuracy                           0.93    175341
   macro avg       0.95      0.89      0.92    175341
weighted avg       0.94      0.93      0.93    175341

TN=44340  FP=11660  FN=279  TP=119062

Normal scores — mean=0.1031  p95=0.4621
Attack scores — mean=0.8233  p05=0.4496

INCIDENT AGENT — MULTICLASS (collapsed to binary)
SOC cost (binary collapse) : 349222
              precision    recall  f1-score   support

      Normal       0.65      0.97      0.78     56000
      Attack       0.98      0.76      0.86    119341

    accuracy                           0.83    175341
   macro avg       0.82      0.86      0.82    175341
weighted avg       0.88      0.83      0.83    175341

TN=54261  FP=1739  FN=28812  TP=90529


In [5]:
print(y_multi.value_counts())

attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
Name: count, dtype: int64


# Testing

In [7]:
print("\n=======================================================")
print("FINAL HOLD-OUT TEST SET EVALUATION")
print("=======================================================\n")

# Load the OOF-optimised threshold and cost constants from disk so this cell
# runs correctly after a kernel restart without needing cells 2-5 in memory.
import json as _json
_threshold_path = "Saves/optimal_threshold.json"
with open(_threshold_path) as _f:
    _threshold_record = _json.load(_f)
optimal_threshold = _threshold_record["optimal_threshold"]
COST_FN = _threshold_record["cost_fn"]
COST_FP = _threshold_record["cost_fp"]
print(f"Loaded threshold: {optimal_threshold:.4f}  (FN×{COST_FN}, FP×{COST_FP})")

test_file_path = r"UNSW-NB15\unsw_test.csv"

# Test mode: prepare_datasets returns X_num already transformed by the saved
# full-train scaler+PCA. X_num_raw is unused in the test path but unpacked
# for API consistency.
X_test_cat, X_test_num, X_test_num_raw, y_test_bin, cat_cols_test, y_test_multi = prepare_datasets(test_file_path, is_train=False)

# Verify PCA dimensionality matches the saved artifact (guards against
# retraining with a different variance threshold or different data)
saved_pca = joblib.load(r"Saves/feature_pca.pkl")
expected_pca_dims = saved_pca.n_components_
assert X_test_num.shape[1] == expected_pca_dims, (
    f"PCA dimensionality mismatch: expected {expected_pca_dims} components, "
    f"got {X_test_num.shape[1]}. Retrain or reload the correct scaler/PCA artifacts."
)

# Load frozen base models
print("\nLoading frozen base models from disk...")
loaded_catboost = CatBoostClassifier()
loaded_catboost.load_model("Saves/catboost_base.cbm")
loaded_rcf = RCF.load_model("Saves/rcf_base.pkl")
loaded_incident = joblib.load("Saves/incident_calibrated.pkl")


# Generate Level-1 base scores
print("Generating base model scores (CatBoost, RCF, Incident)...")
cat_scores_test = loaded_catboost.predict_proba(X_test_cat)[:, 1]
X_test_num_scaled = final_rcf.rcf_scaler.transform(X_test_num_raw)
rcf_scores_test_norm = 1.0 - final_rcf.predict_proba(X_test_num_scaled)

incident_proba_test = loaded_incident.predict_proba(X_test_cat)
incident_entropy_test = _incident_entropy(incident_proba_test)

test_top2_margin = _top2_margin(incident_proba_test)  

print("Fusing scores through the Cost-Sensitive Meta-Learner...")
X_meta_test = np.column_stack((cat_scores_test, rcf_scores_test_norm, incident_entropy_test, test_top2_margin))
loaded_meta_learner = CostSensitiveMetaLearner.load_model("Saves/meta_learner.pkl")
test_final_risk = loaded_meta_learner.predict_proba(X_meta_test)

# Zero Trust boundary — use the OOF-derived threshold, not 0.5
test_predictions = (test_final_risk >= optimal_threshold).astype(int)

# SOC operational metrics
tn, fp, fn, tp = confusion_matrix(y_test_bin, test_predictions).ravel()
final_soc_cost = (fn * COST_FN) + (fp * COST_FP)

print("\n--- FINAL OPERATIONAL REPORT (TEST SET) ---")
print(f"Decision threshold (OOF-optimised): {optimal_threshold:.4f}")
print(classification_report(y_test_bin, test_predictions, target_names=["Normal (0)", "Attack (1)"]))

print("\n--- ZERO TRUST SOC IMPACT ---")
print(f"True Negatives (Traffic Safely Allowed): {tn}")
print(f"True Positives (Attacks Successfully Blocked): {tp}")
print(f"False Positives (Unjustified Alert Fatigue): {fp}")
print(f"False Negatives (Missed Attacks): {fn}")
print("-------------------------------------------------------")
print(f"Total Operational Penalty Score: {final_soc_cost}")


FINAL HOLD-OUT TEST SET EVALUATION

Loaded threshold: 0.7889  (FN×12, FP×2)
Loading and transforming data from UNSW-NB15\unsw_test.csv (Mode: Test)...
Final Categorical features: 6
Numerical features processed: 20 Principal Components

Loading frozen base models from disk...
RCF state successfully loaded from Saves/rcf_base.pkl
Generating base model scores (CatBoost, RCF, Incident)...


RRCF Blind Test Scoring: 100%|██████████| 175341/175341 [4:44:33<00:00, 10.27it/s]       



[RCF] Score distribution — min=0.0050  mean=0.3169  max=1.0000  % at ceiling (≥0.999): 10.3%
[RCF] WARNING: 10.3% of scores saturated at ceiling. Expected ~0.5% (p99.5 ceiling). Consider retraining with more attack samples or checking for distribution shift between train and test.
Fusing scores through the Cost-Sensitive Meta-Learner...
Model state successfully loaded from Saves/meta_learner.pkl

--- FINAL OPERATIONAL REPORT (TEST SET) ---
Decision threshold (OOF-optimised): 0.7889
              precision    recall  f1-score   support

  Normal (0)       0.96      0.80      0.87     56000
  Attack (1)       0.91      0.98      0.95    119341

    accuracy                           0.93    175341
   macro avg       0.93      0.89      0.91    175341
weighted avg       0.93      0.93      0.92    175341


--- ZERO TRUST SOC IMPACT ---
True Negatives (Traffic Safely Allowed): 45033
True Positives (Attacks Successfully Blocked): 117226
False Positives (Unjustified Alert Fatigue): 10967
Fa

In [8]:
# Generate the predicted threat labels here so we don't need the model in Cell 7
print("\nGenerating final threat classification labels...")
predicted_threat_labels = loaded_incident.predict(X_test_cat)

print("--- SAVING EVALUATION CACHE FOR CELL 7 ---")
os.makedirs("Saves", exist_ok=True)
cache_data = {
    "X_test_cat": X_test_cat,
    "X_test_num_raw": X_test_num_raw,
    "y_test_bin": y_test_bin,
    "cat_scores_test": cat_scores_test,
    "rcf_scores_test_norm": rcf_scores_test_norm,
    "test_final_risk": test_final_risk,
    "predicted_threat_labels": predicted_threat_labels,
    "incident_entropy_test": incident_entropy_test
}
joblib.dump(cache_data, "Saves/test_evaluation_cache.joblib")
print("Cache saved to Saves/test_evaluation_cache.joblib. You can now safely restart your kernel.")


Generating final threat classification labels...
--- SAVING EVALUATION CACHE FOR CELL 7 ---
Cache saved to Saves/test_evaluation_cache.joblib. You can now safely restart your kernel.


In [2]:
# -----------------------------------------------------------------------
#  SOAR / Zero Trust Agent 
# -----------------------------------------------------------------------

print("--- LOADING EVALUATION CACHE ---")
try:
    cache_data = joblib.load("Saves/test_evaluation_cache.joblib")
    X_test_cat = cache_data["X_test_cat"]
    X_test_num_raw = cache_data["X_test_num_raw"]
    y_test_bin = cache_data["y_test_bin"]
    cat_scores_test = cache_data["cat_scores_test"]
    rcf_scores_test_norm = cache_data["rcf_scores_test_norm"]
    test_final_risk = cache_data["test_final_risk"]
    predicted_threat_labels = cache_data["predicted_threat_labels"]
    incident_entropy_test = cache_data["incident_entropy_test"]
    print("Cache loaded successfully! Ready for SOAR evaluation.")
except FileNotFoundError:
    print("[ERROR] Cache not found. Please run Cell 6 first to generate the cache.")

soar_agent = ZeroTrustSOARAgent()
editor_agent = PlaybookEditorAgent(soar_agent,vector_memory=soar_agent.vector_memory)
print(f"[DEBUG] Playbook file path: {soar_agent.playbook_file}")

N_SAMPLE = 100

np.random.seed(69)
true_attack_indices = np.where(y_test_bin == 1)[0]
true_normal_indices = np.where(y_test_bin == 0)[0]

sample_indices = np.concatenate([
    np.random.choice(true_attack_indices, N_SAMPLE // 2, replace=False),
    np.random.choice(true_normal_indices, N_SAMPLE // 2, replace=False)
])
np.random.shuffle(sample_indices)

print("=" * 60)
print("SOAR / ZERO TRUST AGENT — SAMPLE EVENT EVALUATION")
print("=" * 60)

soar_decisions = []

run_start_time = datetime.datetime.now().isoformat()

for i, idx in enumerate(sample_indices):
    print(f"\n{'─' * 60}")
    print(f"Event {i + 1}/{len(sample_indices)}  |  Test row index: {idx}")
    print(f"Ground truth: {'ATTACK' if y_test_bin.iloc[idx] == 1 else 'NORMAL'}")

    telemetry_row = {
        **X_test_cat.iloc[idx].to_dict(),
        **X_test_num_raw.iloc[idx].to_dict()
    }

    context = soar_agent.construct_context(
        event_id=idx,
        cat_risk=float(cat_scores_test[idx]),
        rcf_risk=float(rcf_scores_test_norm[idx]),
        final_risk=float(test_final_risk[idx]),
        threat_type=str(predicted_threat_labels[idx]),
        telemetry=telemetry_row,
        incident_entropy=float(incident_entropy_test[idx])
    )

    decision = soar_agent.run(event_id=idx, context=context)
    soar_decisions.append(decision)

    ground_truth = y_test_bin.iloc[idx]
    playbook = decision.get("playbook")

    if ground_truth == 0 and playbook in ["NETWORK_ISOLATION", "STEP_UP_AUTH", "RATE_LIMIT_DOS"]:
        editor_agent.autonomous_fp_correction(
            event_id=idx,
            context=context,
            decision=decision
        )

    elif ground_truth == 1 and playbook == "ALLOW":
        editor_agent.autonomous_fn_correction(
            event_id=idx,
            context=context,
            decision=decision
        )

    # ── NEW: Periodic rule consolidation ──────────────────────────────
    # Every 10 events, ask the LLM to merge any near-duplicate learned
    # rules that have accumulated. Keeps playbooks.json lean and prevents
    # hyper-specific single-event rules from crowding out broader patterns.
    if (i + 1) % 10 == 0:
        editor_agent.consolidate_rules()

# ── NEW: End-of-run rule health report ────────────────────────────────
# Prints any rules that were implicated in FN_THRESHOLD+ false negatives,
# with their full definition and recent incident history.
editor_agent.review_flagged_rules()

# --- Summary ---
print(f"\n{'=' * 60}")
print("SAMPLE EVALUATION SUMMARY")
print(f"{'=' * 60}")

playbook_counts = {}
for d in soar_decisions:
    pb = d.get("playbook", "UNKNOWN")
    playbook_counts[pb] = playbook_counts.get(pb, 0) + 1

for playbook, count in sorted(playbook_counts.items()):
    print(f"  {playbook:<22} : {count} event(s)")

fp_overrides = sum(1 for d in soar_decisions if d.get("is_false_positive"))
print(f"\n  Total False-Positive Overrides (Policy + LLM) : {fp_overrides}")
print(f"  Agent memory written to                       : {soar_agent.memory_file}")

--- LOADING EVALUATION CACHE ---
Cache loaded successfully! Ready for SOAR evaluation.
Creating new configuration file: playbooks.json
Creating new configuration file: agent_memory.json
[PolicyAgent] Creating default policy config: policy_config.json
[ZeroTrustSOARAgent] Loaded optimal threshold: 0.7889 (OOF SOC cost=37532, FN×12 FP×2)
[VectorMemory] Initialising ChromaDB at 'chroma_db' …
[VectorMemory] Loading encoder 'all-MiniLM-L6-v2' …


c:\Users\Arib Ansari\Desktop\Pithun\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[VectorMemory] Ready.  corrections=0  incidents=0  rules=0
[DEBUG] Playbook file path: playbooks.json
SOAR / ZERO TRUST AGENT — SAMPLE EVENT EVALUATION

────────────────────────────────────────────────────────────
Event 1/100  |  Test row index: 111863
Ground truth: NORMAL

[ML Gate]   fused_risk=0.0230 ≤ threshold=0.7889 → Benign, logging only.
[SOAR] Event 111863: RCF score below threshold → LOG_ONLY

────────────────────────────────────────────────────────────
Event 2/100  |  Test row index: 40753
Ground truth: NORMAL

[ML Gate]   fused_risk=0.0250 ≤ threshold=0.7889 → Benign, logging only.
[SOAR] Event 40753: RCF score below threshold → LOG_ONLY

────────────────────────────────────────────────────────────
Event 3/100  |  Test row index: 50606
Ground truth: ATTACK

[ML Gate]   fused_risk=0.8410 > threshold=0.7889 → Escalating to Policy Agent.

[PolicyAgent] Trust Score: 0.7477 | Signature: tcp_-_FIN_Normal_R3_A3_V1_E2
[ZeroTrust]  Trust ACCEPTABLE (≥ 0.4) AND Risk < 0.85 → ALLOW fa

c:\Users\Arib Ansari\Desktop\Pithun\Lib\site-packages\transformers\models\bert\modeling_bert.py:439: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


[VectorMemory] Stored correction analysis for event 50606 → corr_ffb19355482c

[PlaybookEditorAgent] ✓ FN Self-Healing complete. Signature 'tcp_-_FIN_Normal_R3_A3_V1_E2' moved to quarantine.


────────────────────────────────────────────────────────────
Event 4/100  |  Test row index: 107103
Ground truth: ATTACK

[ML Gate]   fused_risk=1.0000 > threshold=0.7889 → Escalating to Policy Agent.

[PolicyAgent] Trust Score: 0.3500 | Signature: unas_-_INT_DoS_R4_A3_V1_E3
              Violations: UNAPPROVED_COMBO: unas_-
[ZeroTrust]  Trust UNACCEPTABLE (< 0.4) → Response Agent
[DEBUG] LLM raw (140 chars): '{"reasoning":"Threat classified as DoS, initiating rate limiting to mitigate attack.","playbook":"RATE_LIMIT_DOS","is_false_positive":false}'
[VectorMemory] Stored incident summary for event 107103 → inc_0199470ce391

[SOAR] Triggering Playbook: RATE_LIMIT_DOS for Event 107103
       AI Reasoning: Threat classified as DoS, initiating rate limiting to mitigate attack.
       Trust Score: 0.3

In [3]:
# -----------------------------------------------------------------------
# SOAR Analytics & Confusion Matrix
# -----------------------------------------------------------------------
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

# 1. Map ground truth and predictions for this specific sample run
true_labels = [y_test_bin.iloc[idx] for idx in sample_indices]

predicted_labels = []
for d in soar_decisions:
    pb = d.get("playbook", "UNKNOWN")
    # If the playbook allowed the traffic, the AI predicted Normal (0)
    if pb in ["ALLOW", "LOG_ONLY"]:
        predicted_labels.append(0)
    # If the playbook escalated, the AI predicted Attack (1)
    else:
        predicted_labels.append(1)

# 2. Calculate core metrics
tn, fp, fn, tp = confusion_matrix(true_labels, predicted_labels).ravel()
acc = accuracy_score(true_labels, predicted_labels)
prec = precision_score(true_labels, predicted_labels, zero_division=0)
rec = recall_score(true_labels, predicted_labels, zero_division=0)

# 3. Calculate Self-Healing stats from the Playbook Editor's log
fp_corrections = 0
fn_corrections = 0
rules_generated = 0

# Filter strictly to corrections that occurred during THIS run using
# the run_start_time stamp set at the top of the evaluation cell.
# This prevents double-counting when correction_log.json accumulates
# records across multiple runs without being deleted.
for record in editor_agent.correction_log:
    corrected_at = record.get("corrected_at", "")
    if corrected_at < run_start_time:
        continue   # record belongs to a previous run — skip
    eid = record.get("event_id")
    gt = y_test_bin.iloc[eid]
    if gt == 0:
        fp_corrections += 1
    elif gt == 1:
        fn_corrections += 1
    if record.get("new_rule") is not None:
        rules_generated += 1

# 4. Print the Dashboard
print("\n" + "="*60)
print("COGNITIVE SOAR — POST-RUN ANALYTICS")
print("="*60)
print(f"Total Events Evaluated : {len(sample_indices)}")

print("\n--- AUTONOMOUS SELF-HEALING METRICS ---")
print(f"False Positives Detected & Audited : {fp_corrections}")
print(f"False Negatives Detected & Audited : {fn_corrections}")
print(f"New Routing Rules Synthesised    : {rules_generated}")

print("\n--- FINAL SOC PERFORMANCE ---")
print(f"Accuracy  : {acc:.2%}")
print(f"Precision : {prec:.2%}  (Confidence when isolating a machine)")
print(f"Recall    : {rec:.2%}  (Detection rate of actual attacks)")

print("\n--- CONFUSION MATRIX ---")
print(f"                 | Predicted NORMAL | Predicted ATTACK |")
print(f"-----------------|------------------|------------------|")
print(f"   Actual NORMAL | TN: {tn:<12} | FP: {fp:<12} |")
print(f"   Actual ATTACK | FN: {fn:<12} | TP: {tp:<12} |")
print("="*60)


COGNITIVE SOAR — POST-RUN ANALYTICS
Total Events Evaluated : 100

--- AUTONOMOUS SELF-HEALING METRICS ---
False Positives Detected & Audited : 4
False Negatives Detected & Audited : 7
New Routing Rules Synthesised    : 9

--- FINAL SOC PERFORMANCE ---
Accuracy  : 86.00%
Precision : 89.13%  (Confidence when isolating a machine)
Recall    : 82.00%  (Detection rate of actual attacks)

--- CONFUSION MATRIX ---
                 | Predicted NORMAL | Predicted ATTACK |
-----------------|------------------|------------------|
   Actual NORMAL | TN: 45           | FP: 5            |
   Actual ATTACK | FN: 9            | TP: 41           |
